# Module 20: Advanced Python & Next Steps

This final module covers type hints, dataclasses, `collections`, `itertools`, regular expressions, a concurrency overview, and where to go next.

## 20.1 Type Hints

Python is dynamically typed, but **type hints** (PEP 484) let you annotate variables and functions for documentation and tooling (mypy, PyCharm).

Type hints are **not enforced at runtime** — they're just hints.

In [ ]:
# Variable annotations
name: str = "Alice"
age: int = 30
score: float = 9.5
items: list[str] = ["a","b","c"]

# Function annotations
def greet(name: str, times: int = 1) -> str:
    """Return a greeting repeated `times` times."""
    return (f"Hello, {name}! " * times).strip()

result: str = greet("Alice", 2)
print(result)

# Complex types
from typing import Optional, Union, Callable, Any

def divide(a: float, b: float) -> Optional[float]:
    """Return a/b or None if b is zero."""
    return a/b if b != 0 else None

def apply(func: Callable[[int], int], value: int) -> int:
    return func(value)

def mixed(x: Union[int, str]) -> str:
    return str(x)

## 20.2 Dataclasses

`@dataclass` auto-generates `__init__`, `__repr__`, `__eq__`, and more:

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float
    
    def distance_to(self, other: "Point") -> float:
        return ((self.x-other.x)**2 + (self.y-other.y)**2)**0.5

p1 = Point(0, 0)
p2 = Point(3, 4)
print(p1)                       # Point(x=0, y=0)  ← auto __repr__
print(p1 == Point(0,0))         # True              ← auto __eq__
print(p1.distance_to(p2))       # 5.0

@dataclass
class Student:
    name: str
    grades: list = field(default_factory=list)   # mutable default
    
    def average(self) -> float:
        return sum(self.grades)/len(self.grades) if self.grades else 0.0

s = Student("Alice")
s.grades.extend([90, 85, 92])
print(s)
print(s.average())

## 20.3 `collections` — Specialized Data Structures

In [ ]:
from collections import Counter, defaultdict, namedtuple, deque

# Counter — count occurrences
words = "the quick brown fox jumps over the lazy dog the fox".split()
counts = Counter(words)
print(counts)
print(counts.most_common(3))   # top 3 most frequent
print(counts["the"])           # 3

# defaultdict — dict with a default value factory
from collections import defaultdict
graph = defaultdict(list)    # missing keys auto-get empty list
graph["a"].append("b")
graph["a"].append("c")
graph["b"].append("d")
print(dict(graph))

# namedtuple — lightweight immutable class
Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
print(p.x, p.y)
print(p)

# deque — efficient double-ended queue
from collections import deque
d = deque([1,2,3], maxlen=5)
d.appendleft(0)    # add to front
d.append(4)        # add to back
print(d)

## 20.4 `itertools` — Building Blocks for Iterators

In [ ]:
import itertools

# chain — combine iterables
result = list(itertools.chain([1,2,3], [4,5], [6]))
print(result)   # [1,2,3,4,5,6]

# product — Cartesian product
combos = list(itertools.product("AB", repeat=2))
print(combos)   # [AA, AB, BA, BB]

# combinations and permutations
print(list(itertools.combinations([1,2,3,4], 2)))
print(list(itertools.permutations([1,2,3], 2)))

# groupby — group consecutive elements
from itertools import groupby
data = [("a",1),("a",2),("b",3),("b",4),("c",5)]
for key, group in groupby(data, key=lambda x: x[0]):
    print(key, list(group))

# islice — slice from any iterable without building a list
first_5_squares = list(itertools.islice((n**2 for n in range(1000)), 5))
print(first_5_squares)

## 20.5 Regular Expressions (Quick Intro)

In [ ]:
import re

text = "Contact us at info@example.com or support@company.org"

# Search for a pattern
match = re.search(r"\w+@\w+\.\w+", text)
if match:
    print("Found:", match.group())

# Find all matches
emails = re.findall(r"\b[\w.-]+@[\w.-]+\.\w+\b", text)
print("All emails:", emails)

# Substitution
censored = re.sub(r"\w+@\w+\.\w+", "[REDACTED]", text)
print(censored)

# Validate a phone number
def is_valid_phone(phone: str) -> bool:
    pattern = r"^\+?\d{1,3}?[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$"
    return bool(re.match(pattern, phone))

print(is_valid_phone("555-867-5309"))   # True
print(is_valid_phone("hello"))          # False

## 20.6 Concurrency — Overview

Python has three main concurrency approaches:

| Approach | Best for | Module |
|----------|----------|--------|
| Threading | I/O-bound tasks (web, file, network) | `threading` |
| Multiprocessing | CPU-bound tasks (computation) | `multiprocessing` |
| Async/await | High-concurrency I/O | `asyncio` |

> **The GIL:** Python's Global Interpreter Lock means only one thread runs Python code at a time. Threading helps I/O-bound work but NOT CPU-bound. Use `multiprocessing` for CPU-intensive tasks.

In [ ]:
import threading
import time

def download(url, delay):
    """Simulate a download."""
    print(f"Starting {url}...")
    time.sleep(delay)   # simulate network delay
    print(f"Finished {url}")

# Sequential — takes sum of all delays
print("=== Sequential ===")
start = time.time()
download("page1.html", 1)
download("page2.html", 1)
download("page3.html", 1)
print(f"Time: {time.time()-start:.1f}s")

# Threaded — runs concurrently
print("\n=== Threaded ===")
start = time.time()
threads = [
    threading.Thread(target=download, args=(f"page{i}.html", 1))
    for i in range(1, 4)
]
for t in threads: t.start()
for t in threads: t.join()   # wait for all to finish
print(f"Time: {time.time()-start:.1f}s")

In [ ]:
import asyncio

async def fetch(url, delay):
    """Async version of a download."""
    print(f"Fetching {url}...")
    await asyncio.sleep(delay)    # non-blocking sleep
    print(f"Done {url}")
    return f"Content of {url}"

async def main():
    # Run all concurrently
    results = await asyncio.gather(
        fetch("page1.html", 1),
        fetch("page2.html", 1),
        fetch("page3.html", 1),
    )
    print("Results:", results)

# Run the async event loop
asyncio.run(main())

## 20.7 Best Practices & Code Quality

In [ ]:
# 1. Use meaningful names
# Bad:
x = [i for i in range(10) if i % 2 == 0]
# Good:
even_numbers = [n for n in range(10) if n % 2 == 0]

# 2. Keep functions small and focused
# Bad: one function that does many things
# Good: each function does ONE thing

# 3. Use type hints
def calculate_tax(price: float, rate: float = 0.1) -> float:
    return price * rate

# 4. Write docstrings for public functions
def fibonacci(n: int) -> int:
    """
    Return the nth Fibonacci number.
    
    Args:
        n: Position in the sequence (0-indexed)
    Returns:
        The nth Fibonacci number
    """
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

# 5. Handle errors explicitly
def safe_sqrt(x: float) -> float:
    if x < 0:
        raise ValueError(f"Cannot take sqrt of negative number: {x}")
    import math
    return math.sqrt(x)

print("Type hints, docstrings, and error handling — hallmarks of quality code!")

## 20.8 Where to Go Next

You've completed the fundamentals through advanced Python! Here's your roadmap:

### Immediate Next Steps
- **Testing:** `pytest`, `unittest` — write tests for everything
- **Virtual environments & packaging:** `pyproject.toml`, `poetry`
- **Type checking:** `mypy` for static analysis

### Choose Your Path

| Path | Key Libraries |
|------|---------------|
| **Web Development** | FastAPI, Django, Flask |
| **Data Science** | NumPy, Pandas, Matplotlib |
| **Machine Learning** | scikit-learn, PyTorch, TensorFlow |
| **Automation** | Selenium, requests, BeautifulSoup |
| **DevOps / Cloud** | boto3, Docker SDK, Ansible |
| **Game Development** | Pygame, Panda3D |

### Resources
- 📚 [docs.python.org](https://docs.python.org) — official docs
- 🧩 [leetcode.com](https://leetcode.com) — practice problems
- 📖 *Fluent Python* by Luciano Ramalho — intermediate/advanced
- 🎓 [realpython.com](https://realpython.com) — tutorials

### Final Project Ideas
1. CLI task manager
2. Web scraper that saves data to CSV
3. REST API client
4. A Python library with tests and documentation

**Congratulations on completing Python Mastery! 🎉🐍**